# 5장 Seaborn 응용: 질문에 맞는 차트 선택하기

『AI 시대의 파이썬 데이터 시각화』 예제 노트북입니다. 셀 첫 줄의 번호(P·C·B)는 책 지면의 코드 번호와 같습니다. 위에서부터 차례로 실행합니다.

## 준비

새 노트북이면 이 장의 코드보다 먼저 아래 준비 셀을 실행합니다(책의 앞 장과 같은 셀).

**코랩이면 이 셀(P02)을, 내 컴퓨터면 다음 셀(P03)을 실행합니다.** 둘 다 실행하지 않습니다.

In [ ]:
# P02 | 한글 글꼴(코랩): 나눔글꼴을 설치하고 Matplotlib에 등록합니다.
!apt-get -qq update
!apt-get -qq install -y fonts-nanum

import matplotlib.pyplot as plt
from matplotlib import font_manager

for path in font_manager.findSystemFonts():
    if 'Nanum' in path:
        font_manager.fontManager.addfont(path)
plt.rcParams['font.family'] = 'NanumGothic'
plt.rcParams['axes.unicode_minus'] = False

**내 컴퓨터(Mac·Windows)면 이 셀을 실행합니다.** Windows는 글꼴 이름을 `'Malgun Gothic'`으로 바꿉니다.

In [ ]:
# P03 | 한글 글꼴(Mac·Windows): 그래프를 그리기 전에 한 번 실행합니다.
import matplotlib.pyplot as plt

plt.rcParams['font.family'] = 'AppleGothic'   # Windows는 'Malgun Gothic'
plt.rcParams['axes.unicode_minus'] = False    # 음수 기호가 깨지지 않게

## 5.1 질문과 데이터 확인

In [ ]:
# C01 | 월별 표: 4장과 같은 지점별 1~4월 매출을 긴 형식으로 만듭니다.
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

monthly = pd.DataFrame({
    'month': [1, 2, 3, 4] * 2,
    'branch': ['A'] * 4 + ['B'] * 4,
    'sales': [100, 120, 150, 130, 70, 90, 100, 120],   # 백만원
})
branch_colors = {'A': '#0072B2', 'B': '#D55E00'}   # 지점 색(4장과 같음)
totals = monthly.groupby('branch', as_index=False)['sales'].sum()
print(totals)

In [ ]:
# C02 | 관측 표: 지점·채널·주마다 광고비와 매출을 적은 가상 관측 48행입니다.
import numpy as np

rng = np.random.default_rng(202605)   # 시드(시작값)를 정해 늘 같은 난수
records = []
for branch in ['A', 'B']:
    for channel in ['Online', 'Store']:
        for week in range(1, 13):
            ad = 6 + 1.5 * week + (3 if channel == 'Online' else 0)
            sales = 30 + 2.8 * ad + (18 if branch == 'A' else 0)
            sales += (9 if channel == 'Store' else 0) + rng.normal(0, 7)
            records.append({'record_id': f'{branch}-{channel}-{week:02d}',
                            'week': week, 'branch': branch,
                            'channel': channel, 'ad_cost': round(ad, 1),
                            'sales': round(sales, 1)})
obs = pd.DataFrame(records)
print(obs.head(3))
print(obs.groupby(['branch', 'channel']).size())

## 5.2 산점도와 집단 구분

In [ ]:
# C03 | 산점도: 한 행의 광고비와 매출을 점 하나로 찍습니다.
fig, ax = plt.subplots(figsize=(6, 2.8))
sns.scatterplot(data=obs, x='ad_cost', y='sales', color='#666666', ax=ax)
ax.set(title='광고비가 큰 관측에서 매출도 큰가?',
       xlabel='광고비 (백만원)', ylabel='매출 (백만원)')
fig.tight_layout()
plt.show()

In [ ]:
# C04 | 추세선: 산점도에 관측 전체의 방향을 요약한 직선을 더합니다.
fig, ax = plt.subplots(figsize=(6, 2.8))
sns.regplot(data=obs, x='ad_cost', y='sales', color='#666666',
            seed=42, ax=ax)              # 신뢰구간 난수 고정
ax.set(title='관측 48개와 추세선',
       xlabel='광고비 (백만원)', ylabel='매출 (백만원)')
fig.tight_layout()
plt.show()

In [ ]:
# C05 | 집단 구분: 같은 지점을 색(hue)과 모양(style)으로 함께 나눕니다.
fig, ax = plt.subplots(figsize=(6, 2.8))
sns.scatterplot(data=obs, x='ad_cost', y='sales', hue='branch',
                style='branch', hue_order=['A', 'B'],
                palette=branch_colors, ax=ax)
ax.set(title='지점을 나눠 다시 본 관계',
       xlabel='광고비 (백만원)', ylabel='매출 (백만원)')
ax.legend(title='지점', loc='lower right')
fig.tight_layout()
plt.show()

In [ ]:
# C06 | 점 겹침: 겹친 점은 그림에서 셀 수 없으므로 표로 셉니다.
dense = pd.concat([obs, obs, obs], ignore_index=True)   # 같은 48행 × 3
print('행 수:', len(dense))
print('서로 다른 좌표:',
      len(dense[['ad_cost', 'sales']].drop_duplicates()))

### 직접 해 보기

**W5-01** C05의 X축을 관측 주로 바꿔, 매출이 늘어난 까닭을 광고비 하나로 말할 수 있는지 따져 봅니다.

1. C05를 복사해 `x='week'`로 바꾸고 X축 라벨을 `관측 주`로, 제목을 알맞게 바꿉니다.
2. 주가 지날수록 두 지점의 매출이 어떻게 달라지는지 숫자와 단위를 넣어 한 문장으로 씁니다.
3. C02의 계산식에서 광고비 `ad`가 주 `week`와 함께 커지는 것을 확인하고, "광고비 때문에 매출이 올랐다"고 할 수 없는 이유를 한 문장으로 씁니다.

**완료 기준**: X축이 관측 주(1~12)이고 A·B 범례가 있으며, 관찰 문장에 숫자와 단위가, 한계 문장에 광고비와 시기가 함께 커졌다는 점이 들어 있다.

In [ ]:
# W5-01 — 여기에서 직접 해 봅니다.

## 5.3 패싯과 반환 객체

In [ ]:
# C07 | 패싯: channel 열의 값마다 패널을 나눠 같은 산점도를 그립니다.
g = sns.relplot(data=obs, x='ad_cost', y='sales', hue='branch',
                style='branch', col='channel',
                col_order=['Online', 'Store'], hue_order=['A', 'B'],
                palette=branch_colors, kind='scatter', height=2.6)
g.set_axis_labels('광고비 (백만원)', '매출 (백만원)')
g.set_titles('{col_name} 채널')
g.legend.set_title('지점')
g.figure.suptitle('채널을 나누어 본 지점별 관계')
g.figure.subplots_adjust(top=0.81)   # 전체 제목 자리
print(type(g).__name__, g.axes.shape)
plt.show()

In [ ]:
# C08 | 반환 객체 수정: 모든 패널에 같은 범위를 주고, 범위 밖 관측을 셉니다.
inside = obs['ad_cost'].between(5, 29) & obs['sales'].between(45, 135)
print('범위 밖 관측 수:', (~inside).sum())
g.set(xlim=(5, 29), ylim=(45, 135))    # 모든 패널에 같은 축 범위

narrow = obs['ad_cost'].between(10, 25)
print('X축을 10~25로 좁히면 숨는 관측 수:', (~narrow).sum())
for ax in g.axes.flat:                  # 패널을 하나씩 꺼내 수정
    ax.grid(axis='y', alpha=0.3)
g.figure

### 직접 해 보기

**W5-02** C07의 역할을 바꿔, 같은 지점 안에서 두 채널을 견줍니다.

1. C07을 복사해 `col='branch'`, `hue='channel'`, `style='channel'`로 바꿉니다. `col_order`·`hue_order`·`palette`는 바꾸기 전 열의 값에 맞춘 것이라 지웁니다.
2. 패널 제목을 `'{col_name}지점'`으로, 범례 제목을 `채널`로, 전체 제목을 알맞게 바꿉니다.
3. C08처럼 범위 밖 관측 수를 출력하고 두 패널에 같은 범위를 줍니다.
4. 같은 지점 안에서 두 채널의 점이 어떻게 다른지 한 문장으로 씁니다.

**완료 기준**: A지점·B지점 두 패널에 두 채널이 나뉘어 있고, 축 범위가 같으며, 패널 제목과 전체 제목이 구분되고, 범위 밖 관측 수와 관찰 문장이 있다.

In [ ]:
# W5-02 — 여기에서 직접 해 봅니다.

## 5.4 비중과 크기 비교

In [ ]:
# C09 | 원 그래프: 두 지점의 1~4월 합계에서 각 지점이 차지하는 비중입니다.
share = totals.assign(
    percent=totals['sales'] / totals['sales'].sum() * 100)   # 비중(%) 열
print(share.round(1))

fig, ax = plt.subplots(figsize=(6, 2.0))
ax.pie(share['sales'], labels=share['branch'], autopct='%1.1f%%',
       colors=[branch_colors[b] for b in share['branch']],
       startangle=90)
ax.set_title('1~4월 두 지점 매출 구성')
plt.show()

In [ ]:
# C10 | 100% 누적 막대: 달마다 두 지점의 비중을 한 축에서 견줍니다.
wide = monthly.pivot(index='month', columns='branch', values='sales')
a_pct = wide['A'] / (wide['A'] + wide['B']) * 100    # A지점 비중(%)

fig, ax = plt.subplots(figsize=(6, 2.8))
ax.bar(wide.index, a_pct, color=branch_colors['A'], label='A')
ax.bar(wide.index, 100 - a_pct, bottom=a_pct,       # A 위에 B를 쌓음
       color=branch_colors['B'], hatch='//', label='B',
       edgecolor='black', linewidth=0)
ax.bar_label(ax.containers[0], fmt='%.0f', label_type='center',
             color='white')
ax.set(title='달마다 두 지점 매출 구성', xlabel='월',
       ylabel='비중 (%)', xlim=(0.4, 5.6))
ax.set_xticks([1, 2, 3, 4])
ax.legend(title='지점', loc='center right')         # 오른쪽 빈자리에
fig.tight_layout()
plt.show()

In [ ]:
# C11 | 원 그래프와 가로 막대: 작은 차이와 순위는 막대 길이로 견줍니다.
product = pd.DataFrame({'product': ['P1', 'P2', 'P3', 'P4', 'P5'],
                        'sales': [24, 23, 21, 18, 14]})   # 백만원
ordered = product.sort_values('sales')   # 작은 값부터 → 큰 막대가 위

fig, axes = plt.subplots(1, 2, figsize=(6, 2.8))
axes[0].pie(product['sales'], labels=product['product'],
            autopct='%1.0f%%', startangle=90)
axes[0].set_title('구성: 전체 100백만원')
bars = axes[1].barh(ordered['product'], ordered['sales'],
                    color='#666666')
axes[1].bar_label(bars, padding=3)
axes[1].set(title='비교: P1과 P2의 차이 1백만원',
            xlabel='매출 (백만원)', xlim=(0, 28))
fig.tight_layout()
plt.show()

### 직접 해 보기

**W5-03** P2의 매출을 26으로 바꾼 상품 표(24, 26, 21, 18, 14)로 "P1과 P2 가운데 어느 상품의 매출이 얼마나 큰가?"에 답할 차트를 고릅니다.

1. C11의 두 차트 중 질문에 맞는 것을 골라 값 라벨·단위를 답니다.
2. P1과 P2의 차이를 따로 계산해 그림과 대조하고 제목에도 씁니다.
3. 고른 차트가 이 질문에 맞는 이유를 한 문장으로 씁니다.

**완료 기준**: 그림에서 두 상품의 순위와 차이가 읽히고(막대라면 0에서 시작), 따로 계산한 차이가 그림·제목과 같으며, 고른 이유를 썼다.

In [ ]:
# W5-03 — 여기에서 직접 해 봅니다.

## 5.5 차트 선택의 검수와 해석

In [ ]:
# C12 | 한 장 보고서: 목적이 다른 두 차트를 나란히 두고 출처를 밝힙니다.
fig, axes = plt.subplots(1, 2, figsize=(6, 3.0))
sns.scatterplot(data=obs, x='ad_cost', y='sales', hue='branch',
                style='branch', hue_order=['A', 'B'],
                palette=branch_colors, s=25, ax=axes[0])
axes[0].set(title='관계: 관측 표 48행', xlabel='광고비 (백만원)',
            ylabel='매출 (백만원)')
axes[0].legend(title='지점', loc='lower right')

ranked = totals.sort_values('sales')        # B(380), A(500) 순서
bars = axes[1].barh(ranked['branch'], ranked['sales'],
                    color=[branch_colors[b] for b in ranked['branch']],
                    hatch=['//', ''], edgecolor='black', linewidth=0)
axes[1].bar_label(bars, padding=3)
axes[1].set(title='크기: 월별 표의 1~4월 합계',
            xlabel='매출 합계 (백만원)', ylabel='지점', xlim=(0, 600))
fig.text(0.99, 0.01, '자료: 교육용 가상 자료(관측 표·월별 표)',
         ha='right', fontsize=9)
fig.tight_layout()
plt.show()

In [ ]:
# C13 | 히트맵: 지점·채널마다 관측당 평균 매출을 색 칸으로 봅니다.
table = obs.pivot_table(index='branch', columns='channel',
                        values='sales', aggfunc='mean')
fig, ax = plt.subplots(figsize=(6, 2.0))
sns.heatmap(table, annot=True, fmt='.1f', cmap='Greys',
            linewidths=0.5, linecolor='#999999', ax=ax)   # 칸 테두리
ax.set(title='지점·채널별 관측당 평균 매출 (백만원)',
       xlabel='채널', ylabel='지점')
fig.tight_layout()
plt.show()

### 직접 해 보기

**W5-04** "A지점은 광고비 덕분에 매출이 증가했다"는 주장을 검토합니다.

1. C13의 `values`를 `'ad_cost'`로 바꿔 지점·채널별 평균 광고비 히트맵을 그리고, 제목도 바꿉니다.
2. 같은 채널 안에서 A지점과 B지점의 평균 광고비와 평균 매출을 견줍니다.
3. 관찰(같은 채널 안 두 지점의 비교 수치 하나 이상), 한계(주장이 자료를 넘어서는 부분 — W5-01의 시기 문제도 포함), 추가 확인(더 필요한 자료나 조사)을 한 문장씩 씁니다.

**완료 기준**: 광고비 히트맵을 그렸고, 관찰 문장에 같은 채널 안 두 지점의 수치와 단위가 들어 있으며, 한계와 추가 확인 문장이 있다.

In [ ]:
# W5-04 — 여기에서 직접 해 봅니다.

## 5.6 해석과 점검

## 직접 해 보기 풀이

먼저 위의 연습 칸에서 직접 해 본 뒤에 봅니다. 풀이는 한 가지 예이고, 완료 기준을 만족하면 다른 코드도 됩니다.

### W5-01

In [ ]:
# W5-01 | 풀이: C05의 X축을 관측 주로 바꿉니다.
fig, ax = plt.subplots(figsize=(6, 2.8))
sns.scatterplot(data=obs, x='week', y='sales', hue='branch',
                style='branch', hue_order=['A', 'B'],
                palette=branch_colors, ax=ax)
ax.set(title='관측 주가 지날수록 매출도 큰가?',
       xlabel='관측 주', ylabel='매출 (백만원)')
ax.legend(title='지점', loc='lower right')
fig.tight_layout()
plt.show()

In [ ]:
# 2·3. 문장에 쓸 값: 주별 평균 매출, 주별 광고비
print(obs.groupby(['branch', 'week'])['sales'].mean().unstack()[[1, 12]].round(1))
print(obs.groupby('week')['ad_cost'].mean()[[1, 12]])
weeks = ax.collections[0].get_offsets()[:, 0]
assert weeks.min() == 1 and weeks.max() == 12
# 관찰(예): 1주에서 12주로 가며 두 채널 평균 매출이 A지점은 79.2에서 120.4백만원, B지점은 53.6에서 102.4백만원으로 커졌다.
# 한계(예): C02의 계산식에서 광고비가 주마다 1.5백만원씩 함께 커지므로(1주 9.0 → 12주 25.5백만원),
#   매출이 커진 것이 광고비 때문인지 시기 때문인지 이 자료로는 가를 수 없다.

### W5-02

In [ ]:
# W5-02 | 풀이: C07의 역할을 바꿔, 지점마다 패널을 나누고 두 채널을 견줍니다.
g = sns.relplot(data=obs, x='ad_cost', y='sales', col='branch',
                hue='channel', style='channel', kind='scatter', height=2.6)
g.set_axis_labels('광고비 (백만원)', '매출 (백만원)')
g.set_titles('{col_name}지점')
g.legend.set_title('채널')
g.figure.suptitle('지점 안에서 본 채널별 관계')
g.figure.subplots_adjust(top=0.81)

In [ ]:
# 3. 범위 밖 관측 수와 같은 범위
inside = obs['ad_cost'].between(5, 29) & obs['sales'].between(45, 135)
print('범위 밖 관측 수:', (~inside).sum())
g.set(xlim=(5, 29), ylim=(45, 135))         # 두 패널에 같은 범위
print(obs.groupby(['branch', 'channel'])[['ad_cost', 'sales']].mean().round(1))
assert [ax.get_xlim() for ax in g.axes.flat] == [(5, 29), (5, 29)]
g.figure                                      # 앞 셀의 그림을 다시 보입니다(C08)
# 관찰(예): 같은 지점 안에서 Online 점은 광고비가 평균 3백만원 큰 오른쪽(18.8 대 15.8백만원)에 놓이지만,
#   평균 매출은 두 채널이 비슷하다(A지점 101.5 대 102.0백만원).

### W5-03

In [ ]:
# W5-03 | 풀이: 두 상품의 차이를 묻는 질문이라 C11의 가로 막대를 골라 다시 그립니다.
product2 = pd.DataFrame({'product': ['P1', 'P2', 'P3', 'P4', 'P5'],
                         'sales': [24, 26, 21, 18, 14]})   # 백만원
ordered2 = product2.sort_values('sales')                     # 작은 값부터 → 큰 막대가 위

fig, ax = plt.subplots(figsize=(6, 2.6))
bars = ax.barh(ordered2['product'], ordered2['sales'], color='#666666')
ax.bar_label(bars, padding=3)
ax.set(title='상품별 매출: P2가 P1보다 2백만원 많다',
       xlabel='매출 (백만원)', xlim=(0, 30))
fig.tight_layout()
plt.show()

In [ ]:
# 3. 따로 계산해 그림과 대조
p1 = product2.loc[product2['product'] == 'P1', 'sales'].iloc[0]
p2 = product2.loc[product2['product'] == 'P2', 'sales'].iloc[0]
print('P2 − P1 =', p2 - p1, '백만원')
assert ax.get_xlim()[0] == 0 and list(ordered2['product'])[-1] == 'P2'
# 이유(예): 24와 26백만원의 차이는 원 그래프의 조각 각도로는 가늠하기 어렵지만, 0에서 시작하는 막대 길이로는 바로 보인다.

### W5-04

In [ ]:
# W5-04 | 풀이: C13의 values를 광고비로 바꿔 평균 광고비 히트맵을 그립니다.
table_ad = obs.pivot_table(index='branch', columns='channel',
                           values='ad_cost', aggfunc='mean')
fig, ax = plt.subplots(figsize=(6, 2.0))
sns.heatmap(table_ad, annot=True, fmt='.1f', cmap='Greys',
            linewidths=0.5, linecolor='#999999', ax=ax)
ax.set(title='지점·채널별 관측당 평균 광고비 (백만원)',
       xlabel='채널', ylabel='지점')
fig.tight_layout()
plt.show()

In [ ]:
# 2. 같은 채널 안에서 두 지점의 평균 광고비와 평균 매출
print(table_ad.round(1))
print(obs.pivot_table(index='branch', columns='channel', values='sales', aggfunc='mean').round(1))
# 관찰(예): Online 채널에서 두 지점의 평균 광고비는 18.8백만원으로 같은데, 평균 매출은 A지점 101.5, B지점 81.1백만원이다.
# 한계(예): 광고비가 같아도 A지점 매출이 크므로 차이를 광고비로 설명할 수 없고, 광고비는 주와 함께 커져(W5-01) 시기의 몫도 가를 수 없다.
# 추가 확인(예): 같은 주·같은 광고비에서 두 지점을 견주거나, 지점의 입지·고객 수 같은 다른 자료를 본다.